In [1]:
# prvotni import

import torch
from torch.autograd import grad
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR
import sys
from numpy import pi
sys.path.append('../..')

In [2]:
# vlastni import
from src import train, utils
from src import calculus as calc
import src.data.cube_domain as cb
import src.models.mlp_model as mm

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# konstanty (parametry problemu)
ALPHA = 1.14
A = 2
B = 2
T_MAX = 0.01

In [6]:
l_bounds = [0, 0, 0]
u_bounds = [A, B, T_MAX]

plot_ctx = utils.PlotContext(
    l_bounds=l_bounds,
    u_bounds=u_bounds,
    patches=[],
    colour_map='inferno',
    vmin=-3,
    vmax=4,
    device=device,
    N=100,
    save_img=False
)

In [7]:
def initial_condition(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(pi * x[:, 0:1] / A) * torch.sin(pi * x[:, 1:2] / B) - 3 * torch.sin(5 * pi * x[:, 0:1] / A) * torch.sin(5 * pi * x[:, 1:2] / B)

def analytical_solution(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(pi * x[:, 0:1] / A) * torch.sin(pi * x[:, 1:2] / B) * torch.exp(-ALPHA * pi**2 * x[:, 2:3] * (A**2 + B**2) / ((A * B)**2)) +\
           (-3) * torch.sin(5 * pi * x[:, 0:1] / A) * torch.sin(5 * pi * x[:, 1:2] / B) * torch.exp(-ALPHA * 25 * pi**2 * x[:, 2:3] * (A**2 + B**2) / ((A * B)**2))

#utils.plot_function_on_2d_cube(lambda x: initial_condition(torch.cat((x[:, 0:2], torch.zeros((x.shape[0], 1))), dim=1)), plot_ctx)

In [8]:
def pde_residuum(pde_input: torch.Tensor, model: torch.nn.Module) -> torch.Tensor:
    # PDE ztrata:
    pde_output = model(pde_input)
    u_t = grad(pde_output, pde_input, torch.ones_like(pde_output), create_graph=True)[0][:, -1:]
    #print(u_t)
    laplacian = calc.laplacian(pde_input, pde_output, device=device)
    return u_t - ALPHA * laplacian

def loss_fn(model: torch.nn.Module, domain: cb.CubeDomain) -> torch.Tensor:
    # PDE ztrata:
    pde_input = domain.interior.requires_grad_(True)
    pde_res = pde_residuum(pde_input, model)
    pde_loss = torch.mean((pde_res)**2)

    # ztrata na hranicich:
    side_input = domain.get_side_points(2).requires_grad_(True)
    side_output = model(side_input)
    side_loss = torch.mean(side_output**2)

    # ztrata na pocatku
    initial_input = domain.sides[-1][0].requires_grad_(True)
    init_output = model(initial_input)
    init_values = initial_condition(initial_input)
    init_loss = torch.mean((init_output - init_values)**2)

    return [pde_loss, 100 * side_loss, 1000 * init_loss]

# ztrata pomoci L2 normy
def loss_l2(model: torch.nn.Module, domain: cb.CubeDomain) -> torch.Tensor:
    return calc.L2_norm(model, analytical_solution, 3, u_bounds, l_bounds, device, 50)

In [9]:
model_ctx = mm.ModelContext(
    l_bounds=l_bounds,
    u_bounds=u_bounds,
    input_dim=3,
    output_dim=1,
    layer=[64, 64, 64],
    fourier_features=True,
    fourier_frequencies=128,
    fourier_scale=1.0,
    fourier_separator=1
)

model = mm.MLPModel(model_ctx).to(device=device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, patience=200, factor=0.75)

In [10]:
domain_ctx = cb.CubeContext(
    l_bounds=l_bounds,
    u_bounds=u_bounds,
    dim=3,
    N_int=2_000,
    N_sides=[(100, 100), (100, 100), (1000, 100)],
    int_sampling='Latin',
    device=device,
)

domain = cb.CubeDomain(domain_ctx)

In [11]:
train_ctx = train.TrainingContext(
    model=model,
    optimizer=optimizer,
    domain=domain,
    loss_fn=loss_fn,
    epochs=50_000,
    monitor_lr=True,
)

total_loss_values, component_loss_values = train.train_switch_to_lbfgs(train_ctx, lbfgs_lr=0.1, epochs_with_lbfgs=500)

/home/berva/miniconda3/envs/torch/lib/python3.12/site-packages/torch/autograd/graph.py:825: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /opt/conda/conda-bld/libtorch_1745854776362/work/aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Loss at epoch 1 is: 5021.29296875. Current learing rate: 0.001 
Loss at epoch 100 is: 2297.10986328125. Current learing rate: 0.001 
Loss at epoch 200 is: 2071.2666015625. Current learing rate: 0.001 
Loss at epoch 300 is: 2259.7568359375. Current learing rate: 0.001 


KeyboardInterrupt: 

In [ ]:
plot_ctx.x_label = 'Epoch'
plot_ctx.y_label = 'Loss'

plot_ctx.titles = ['Loss training']
utils.plot_loss_values({'Loss': total_loss_values}, plot_ctx)
plot_ctx.titles = ['Loss components (lbfgs epochs not included)']
utils.plot_loss_values({'PDE loss': component_loss_values[0], 'Side loss': component_loss_values[1], 'Initial loss': component_loss_values[2]}, plot_ctx)

In [ ]:
def solution_in_time(function, x: torch.Tensor, time: float) -> torch.Tensor:
    time_tensor = torch.full((x.shape[0], 1), time, device=device)
    input_ = torch.cat((x, time_tensor), dim=1)
    return function(input_)


times = [0.0, 0.001, 0.005, 0.01]
plot_ctx.function_names = ["u", "u"]
plot_ctx.figsize = (18, 6)
plot_ctx.x_label = 'x'
plot_ctx.y_label = 'y'
plot_ctx.fontsize = 16
plot_ctx.vmin = -3
plot_ctx.vmax = 4


for time in times:
    plot_ctx.titles = [f"Model prediction at t={time}", f'Exact solution at t={time}']
    utils.plot_function_on_2d_cube([lambda x: solution_in_time(model, x, time), 
                                    lambda x: solution_in_time(analytical_solution, x, time)], plot_ctx)

In [ ]:
plot_ctx.vmin = 0.0
plot_ctx.vmax = 0.1
plot_ctx.figsize = (6, 4)
plot_ctx.fontsize = 12
for time in times:
    plot_ctx.titles = [f"Absolute error at t={time}"]
    utils.plot_function_on_2d_cube([lambda x: torch.abs(solution_in_time(analytical_solution, x, time) -\
                                     solution_in_time(model, x, time))], plot_ctx)

In [ ]:
plot_ctx.vmin = 0.0
plot_ctx.vmax = 0.1
plot_ctx.figsize = (18, 6)
plot_ctx.fontsize = 12
plot_ctx.titles = [f"Absolute error at t=0.0", f"Absolute error at t=0.01"]
plot_ctx.function_names = ['Err', 'Err']
utils.plot_function_on_2d_cube([lambda x: torch.abs(solution_in_time(analytical_solution, x, 0) -\
                                solution_in_time(model, x, 0)),
                                lambda x: torch.abs(solution_in_time(analytical_solution, x, 0.01) -\
                                solution_in_time(model, x, 0.01))], plot_ctx)